# Phase 2: Preprocessing & Minority-Class Augmentation

| | |
|---|---|
| **Group** | Group 2 |
| **Members** | Evan John Tomy (8884866), Jerin Pious (add student ID) |
| **Program** | Bachelor of Computer Science |
| **Course** | Advanced Topics in Artificial Intelligence and Machine Learning |
| **Course Code** | PROG74040, Spring 2026, Section 1 |
| **Date** | August 10, 2026 |

---

**Notebook 2 of 5** reads the raw training file and writes `train_clean.csv`, `val_clean.csv`, and `pos_weight.pt`, which every notebook after this one depends on. Runs entirely on the local RTX 3060 GPU, with no Colab dependency.

## Purpose
Turns the raw Kaggle comments into a clean, tokenizer-ready dataset: strips HTML and Wikipedia markup, normalizes Unicode, builds a properly stratified train/validation split across all six labels at once, and computes per-label class weights to counter the dataset's imbalance. The second half of this notebook implements and rigorously tests the data-augmentation technique proposed in Phase 1 (back-translation + paraphrasing); the honest result, tested end-to-end in Notebooks 3-5, is that it measurably **hurts** performance, and the final pipeline reverts to the un-augmented data.

In [1]:
import re
import html
import unicodedata

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

train = pd.read_csv("train.csv/train.csv")
print("Loaded:", train.shape)

Loaded: (159571, 8)


In [2]:
def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)                # HTML tags
    text = re.sub(r"==+.*?==+", " ", text)               # == headings ==
    text = re.sub(r"\[\[.*?\]\]", " ", text)             # [[wiki links]]
    text = re.sub(r"\{\{.*?\}\}", " ", text)             # {{templates}}
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["comment_text"].apply(clean_text)

print("=== before / after examples ===")
for _, row in train[["comment_text", "clean_text"]].sample(3, random_state=7).iterrows():
    print("RAW:  ", row["comment_text"].replace("\n", " ")[:150])
    print("CLEAN:", row["clean_text"][:150])
    print()

=== before / after examples ===
RAW:   "==Nagisa Oshima TV documentaries== Hi, I think the Nagisa Oshima TV documentary titles in English are not ""AKA"" but actually just translations by W
CLEAN: " Hi, I think the Nagisa Oshima TV documentary titles in English are not ""AKA"" but actually just translations by Wikipedians. So they should go in b

RAW:   " Same here, he he  I did not notice. Thank you for creating such useful bots! — Blue  formerly iDosh "
CLEAN: " Same here, he he I did not notice. Thank you for creating such useful bots! — Blue formerly iDosh "

RAW:   Why because there was an attack on this article. From then onwards using the PIB report a lot of edits were made which never featured in any of those 
CLEAN: Why because there was an attack on this article. From then onwards using the PIB report a lot of edits were made which never featured in any of those 



In [3]:
still_has_markup = train["clean_text"].str.contains(r"(==|\[\[|\{\{)", regex=True, na=False)
still_has_html = train["clean_text"].str.contains(r"<[^>]+>", regex=True, na=False)
print(f"Rows still containing wiki markup after cleaning: {still_has_markup.sum()} ({still_has_markup.mean()*100:.3f}%)")
print(f"Rows still containing HTML tags after cleaning:   {still_has_html.sum()} ({still_has_html.mean()*100:.3f}%)")
print(f"Empty strings after cleaning: {(train['clean_text'].str.len() == 0).sum()}")

C:\Users\evanj\AppData\Local\Temp\ipykernel_35248\2291841291.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  still_has_markup = train["clean_text"].str.contains(r"(==|\[\[|\{\{)", regex=True, na=False)


Rows still containing wiki markup after cleaning: 2242 (1.405%)
Rows still containing HTML tags after cleaning:   0 (0.000%)
Empty strings after cleaning: 25


**Cleaning results**: HTML tags are fully eliminated (0.000% remain). Wiki markup drops from the levels found in Notebook 1 (1.89% train / 29.80% test) down to 1.405% residual, not perfect, likely nested or malformed markup patterns the regex doesn't catch, but a large reduction and acceptable to leave as-is. Twenty-five comments turned out to be *entirely* markup with no actual text underneath (e.g. a bare `==Heading==`), and were dropped in the next cell; feeding an empty string into a tokenizer would be a real bug, not a data point.

In [4]:
before = len(train)
train = train[train["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Dropped {before - len(train)} rows that were entirely markup/HTML (empty after cleaning)")
print("Remaining:", train.shape)

Dropped 25 rows that were entirely markup/HTML (empty after cleaning)
Remaining: (159546, 9)


In [5]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

sample_text = train["clean_text"].iloc[0]
sample_enc = tokenizer(sample_text, truncation=True, padding="max_length", max_length=256)
print("Sample text:", sample_text[:100])
print("input_ids length:", len(sample_enc["input_ids"]))
print("attention_mask length:", len(sample_enc["attention_mask"]))

token_counts = train["clean_text"].sample(2000, random_state=42).apply(
    lambda t: len(tokenizer.encode(t, truncation=False))
)
pct_truncated = (token_counts > 256).mean() * 100
print(f"\nOn a 2,000-row sample: {pct_truncated:.1f}% of comments exceed 256 tokens and would be truncated")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (853 > 512). Running this sequence through the model will result in indexing errors


Sample text: Explanation Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't 
input_ids length: 256
attention_mask length: 256



On a 2,000-row sample: 7.3% of comments exceed 256 tokens and would be truncated


**On the 256-token cutoff**: 7.3% of comments get truncated at this length, a small but real minority losing some content. This matches the Phase 1 proposal's chosen sequence length exactly; the tradeoff (losing the tail of the longest 7% of comments vs. quadrupling memory/compute cost to accommodate them) is a deliberate, documented decision, not an oversight.

In [6]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(msss.split(train, train[LABEL_COLS]))
X_train = train.iloc[train_idx].reset_index(drop=True)
X_val = train.iloc[val_idx].reset_index(drop=True)

print("Train:", X_train.shape, "Val:", X_val.shape)
print(f"\n{'label':15s} {'train_%':>10s} {'val_%':>10s}")
for label in LABEL_COLS:
    print(f"{label:15s} {X_train[label].mean()*100:9.3f}% {X_val[label].mean()*100:9.3f}%")
print("\nTrue joint multi-label stratification (iterative-stratification), balancing all six\n"
      "labels simultaneously rather than `toxic` alone as in the earlier version of this cell.")

Train: (135614, 9) Val: (23932, 9)

label              train_%      val_%
toxic               9.582%     9.581%
severe_toxic        0.999%     0.999%
obscene             5.292%     5.294%
threat              0.299%     0.301%
insult              4.935%     4.935%
identity_hate       0.880%     0.882%

True joint multi-label stratification (iterative-stratification), balancing all six
labels simultaneously rather than `toxic` alone as in the earlier version of this cell.


**What changed here, and why it took a long time to get right**: this split originally stratified on `toxic` alone and simply hoped the other five labels balanced out as a side effect (they did, by a small margin). This version uses `iterative-stratification` to balance all six labels simultaneously: every label now matches to within 0.01-0.07 percentage points between train and val, tighter than the earlier approach achieved. Worth noting for reproducibility: this specific library's algorithm does not scale efficiently to a dataset this size (it processes rows one at a time in a Python loop rather than in bulk), and this cell took roughly 55 minutes to run as a result, confirmed by directly inspecting the library's source code, not just observed slowness. It completed successfully and the result above is correct; a faster hand-rolled alternative (stratifying on the combined six-label pattern directly via `sklearn`) would be the practical choice if this notebook needs to run again.

In [7]:
pos_weights = []
for col in LABEL_COLS:
    pos = X_train[col].sum()
    neg = len(X_train) - pos
    pos_weights.append(neg / pos)

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float)
print(pd.DataFrame({"label": LABEL_COLS, "pos_weight": [round(w, 2) for w in pos_weights]}))
print("\nUse as: torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(device))")

           label  pos_weight
0          toxic        9.44
1   severe_toxic       99.08
2        obscene       17.90
3         threat      333.02
4         insult       19.27
5  identity_hate      112.58

Use as: torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(device))


**What these weights actually do**: `pos_weight` tells the loss function how much more to penalize a *missed* positive example of each label. `threat` at 333.02 means missing an actual threat is penalized over 300 times more heavily than missing a non-threat, a direct, mathematical response to `threat` being ~330 times rarer than it is common. This is the mechanism (not augmentation) that does the real work of imbalance handling in this project, used identically by the LSTM and DistilBERT in Notebooks 3 and 4.

In [8]:
X_train.to_csv("train_clean.csv", index=False)
X_val.to_csv("val_clean.csv", index=False)
torch.save(pos_weight_tensor, "pos_weight.pt")
print("Saved train_clean.csv ", X_train.shape)
print("Saved val_clean.csv   ", X_val.shape)
print("Saved pos_weight.pt   ", pos_weight_tensor.tolist())

Saved train_clean.csv  (135614, 9)
Saved val_clean.csv    (23932, 9)
Saved pos_weight.pt    [9.436662673950195, 99.0841293334961, 17.895639419555664, 333.0246276855469, 19.265092849731445, 112.5795669555664]


## Summary (core pipeline)
`train_clean.csv`, `val_clean.csv`, and `pos_weight.pt` are the canonical inputs for every notebook that follows. The rest of this notebook documents a full, honest experiment in data augmentation, implemented exactly as Phase 1 proposed, tested rigorously, and reverted after being shown to hurt results. That negative result is preserved below and in `augmentation_experiment_results/`.

## Data Augmentation for Minority Classes

Per Table 2 of the Phase 1 proposal: back-translation (EN to FR to EN, via Helsinki-NLP MarianMT) + paraphrasing (via a T5 paraphrase model), both free and local, with no API keys and no cost. Targets `threat`, `severe_toxic`, and `identity_hate`, the three weakest-performing labels identified in `05_evaluation_comparison.ipynb`. Only `X_train` (the training split) is augmented; `X_val` stays untouched, since augmenting validation data would defeat its purpose as an honest check.

In [9]:
MINORITY_LABELS = ["threat", "severe_toxic", "identity_hate"]

minority_mask = X_train[MINORITY_LABELS].sum(axis=1) > 0
minority_rows = X_train[minority_mask].reset_index(drop=True)
print(f"Minority-class rows to augment: {len(minority_rows):,} of {len(X_train):,}")
print(minority_rows[MINORITY_LABELS].sum())

Minority-class rows to augment: 2,539 of 135,614
threat            406
severe_toxic     1355
identity_hate    1194
dtype: int64


In [10]:
from transformers import MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

en_fr_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
en_fr_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-fr").to(device)
fr_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
fr_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-fr-en").to(device)

def back_translate_batch(texts, batch_size=32, max_length=128):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        fr_enc = en_fr_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            fr_ids = en_fr_model.generate(**fr_enc, max_length=max_length)
        fr_texts = en_fr_tok.batch_decode(fr_ids, skip_special_tokens=True)

        en_enc = fr_en_tok(fr_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            en_ids = fr_en_model.generate(**en_enc, max_length=max_length)
        results.extend(fr_en_tok.batch_decode(en_ids, skip_special_tokens=True))
    return results

backtranslated_texts = back_translate_batch(minority_rows["clean_text"].tolist())
print(f"Back-translated {len(backtranslated_texts):,} rows")
for orig, bt in list(zip(minority_rows["clean_text"], backtranslated_texts))[:2]:
    print("ORIG:", orig[:120])
    print("BT:  ", bt[:120])
    print()

C:\Users\evanj\AppData\Local\Programs\Python\Python314\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Back-translated 2,539 rows
ORIG: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK
BT:   COCKSUCKER BEFORE YOU PISS ON MY WORK

ORIG: You are gay or antisemmitian? Archangel WHite Tiger Meow! Greetingshhh! Uh, there are two ways, why you do erased my com
BT:   Are you gay or anti-Semitism? Archangel WHite Tiger Meow! Salutingshhh! Uh, there are two ways, why you erase my comment



In [11]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

paraphrase_name = "humarin/chatgpt_paraphraser_on_T5_base"
paraphrase_tok = T5Tokenizer.from_pretrained(paraphrase_name)
paraphrase_model = T5ForConditionalGeneration.from_pretrained(paraphrase_name).to(device)

def paraphrase_batch(texts, batch_size=32, max_length=128):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = ["paraphrase: " + t for t in texts[i:i + batch_size]]
        enc = paraphrase_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            out_ids = paraphrase_model.generate(**enc, max_length=max_length, num_beams=4)
        results.extend(paraphrase_tok.batch_decode(out_ids, skip_special_tokens=True))
    return results

paraphrased_texts = paraphrase_batch(minority_rows["clean_text"].tolist())
print(f"Paraphrased {len(paraphrased_texts):,} rows")
for orig, pp in list(zip(minority_rows["clean_text"], paraphrased_texts))[:2]:
    print("ORIG:", orig[:120])
    print("PARA:", pp[:120])
    print()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Paraphrased 2,539 rows
ORIG: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK
PARA: It's important to avoid kissing before going to work.

ORIG: You are gay or antisemmitian? Archangel WHite Tiger Meow! Greetingshhh! Uh, there are two ways, why you do erased my com
PARA: Are you a gay or anti-semite?



**A first hint of trouble, visible in these two examples already**: back-translation turned all-caps shouting into softer punctuation but kept the words themselves; the paraphrase model went further, changing *"before you piss around on my work"* into *"before going to work"*, a genuine meaning shift, not just a reword. Whether this kind of drift actually hurts the labels these rows carry is tested properly, end-to-end, in the outcome section below.

In [12]:
bt_rows = minority_rows.copy()
bt_rows["clean_text"] = backtranslated_texts
bt_rows["comment_text"] = backtranslated_texts

pp_rows = minority_rows.copy()
pp_rows["clean_text"] = paraphrased_texts
pp_rows["comment_text"] = paraphrased_texts

augmented_rows = pd.concat([bt_rows, pp_rows], ignore_index=True)
before_dedup = len(augmented_rows)
augmented_rows = augmented_rows[augmented_rows["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Dropped {before_dedup - len(augmented_rows)} empty-after-generation rows")

train_augmented = pd.concat([X_train, augmented_rows], ignore_index=True)
print(f"\nOriginal train: {len(X_train):,} -> Augmented train: {len(train_augmented):,} (+{len(train_augmented) - len(X_train):,})")
print("\nMinority label counts, before vs after augmentation:")
print(pd.DataFrame({
    "before": X_train[MINORITY_LABELS].sum(),
    "after": train_augmented[MINORITY_LABELS].sum(),
}))

Dropped 0 empty-after-generation rows



Original train: 135,614 -> Augmented train: 140,692 (+5,078)

Minority label counts, before vs after augmentation:
               before  after
threat            406   1218
severe_toxic     1355   4065
identity_hate    1194   3582


In [13]:
pos_weights_aug = []
for col in LABEL_COLS:
    pos = train_augmented[col].sum()
    neg = len(train_augmented) - pos
    pos_weights_aug.append(neg / pos)

pos_weight_aug_tensor = torch.tensor(pos_weights_aug, dtype=torch.float)
print(pd.DataFrame({
    "label": LABEL_COLS,
    "pos_weight_original": [round(w, 2) for w in pos_weights],
    "pos_weight_augmented": [round(w, 2) for w in pos_weights_aug],
}))

train_augmented.to_csv("train_augmented.csv", index=False)
torch.save(pos_weight_aug_tensor, "pos_weight_augmented.pt")
print("\nSaved train_augmented.csv", train_augmented.shape)
print("Saved pos_weight_augmented.pt", pos_weight_aug_tensor.tolist())

           label  pos_weight_original  pos_weight_augmented
0          toxic                 9.44                  6.88
1   severe_toxic                99.08                 33.61
2        obscene                17.90                 11.50
3         threat               333.02                114.51
4         insult                19.27                 12.10
5  identity_hate               112.58                 38.28



Saved train_augmented.csv (140692, 9)
Saved pos_weight_augmented.pt [6.883671283721924, 33.61057662963867, 11.504843711853027, 114.51067352294922, 12.099813461303711, 38.27750015258789]


### Augmentation Experiment Outcome (tested end-to-end, then reverted)

`train_augmented.csv` was tested for real: TF-IDF+LogReg, LSTM+GloVe, and DistilBERT were all retrained on it and re-evaluated on the untouched Kaggle test set (`05_evaluation_comparison.ipynb`). Result: **macro F1 dropped for every model** (DistilBERT 0.560 to 0.511, TF-IDF 0.422 to 0.373, LSTM 0.265 to 0.264), and DistilBERT's `severe_toxic` recall collapsed from 0.802 to 0.457.

**Diagnosis**: back-translation and paraphrasing preserve general hostility (so `toxic`/`obscene`/`insult` stayed flat) but erode the exact, narrow phrasing that `threat`/`severe_toxic`/`identity_hate` depend on to be true: e.g., back-translation turned *"go back to ching chong land"* into *"go back to Chong's land"*, softening the identity-targeted slur while the label stayed `identity_hate=1`. The same three classes degraded across three completely different model architectures (linear, recurrent, transformer), which points at the training data itself rather than any one model's limitation. `severe_toxic`'s ROC-AUC also fell (0.988 to 0.975) with both precision and recall dropping together, a genuine quality regression, not a threshold artifact.

**Decision**: the final pipeline (notebooks 03-05) reverts to `train_clean.csv` / `pos_weight.pt` (pre-augmentation). This experiment's models, metrics, and plots are preserved in `augmentation_experiment_results/` for the report's Technical Depth / error-analysis section, and implementing the Phase 1 augmentation plan and rigorously showing it doesn't help here is itself a real, documented finding.